<a href="https://colab.research.google.com/github/Elchegue64/TAREAS-SSF/blob/T6/T6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import sympy as sp
import numpy as np


# Funciones a integrar


Funciones = [
    "((math.e)**(-x**2/2))/((math.sqrt(2*math.pi)))",
    "(((math.e)**math.sin(x))*((1+x)**2))/((x**2)+1)",
    "((math.e)**x)/((1+(math.e)**x)/1)"
]

# Función segura (evita división entre 0)


def safe_eval(f, x):
    try:
        return f(x)
    except ZeroDivisionError:
        return 0


# Regla del trapecio


def Int_Trap(f, b, a, n):
    h = (b - a)/float(n)
    I = (f(a) + f(b))/2
    x = a
    for k in range(1, n):
        x = a + k*h
        I += safe_eval(f, x)
    return I*h


# Regla de Simpson


def Int_Simpson(f, b, a, n):
    h = (b - a)/float(n)
    I = (f(a) + f(b))
    x = a
    for k in range(1, n):
        x = a + k*h
        if k % 2 == 0:
            I += 2 * safe_eval(f, x)
        else:
            I += 4 * safe_eval(f, x)
    return I * (h/3)


# Cuadratura de Gauss-Legendre


def gauss_legendre(f, b, a, n):
    Xw, w = np.polynomial.legendre.leggauss(n)
    xm = 0.5 * (b + a)
    xr = 0.5 * (b - a)
    I = 0
    for i in range(n):
        I += w[i] * f(xm + xr * Xw[i])
    return I * xr


# Cálculo para cada función


for i in range(len(Funciones)):

    expr_str = (Funciones[i]
        .replace("math.e", "E")
        .replace("math.pi", "pi")
        .replace("math.sqrt", "sqrt")
        .replace("math.sin", "sin")
        .replace("math.cos", "cos")
    )

    # Expresión simbólica
    f_expr = sp.sympify(expr_str, locals={"E": sp.E, "pi": sp.pi, "sqrt": sp.sqrt, "sin": sp.sin, "cos": sp.cos})
    x = sp.Symbol('x')
    f = sp.lambdify(x, f_expr, modules=["math"])

    # Intervalo
    a = -1
    b = 1
    n = 8

    # Derivadas para errores teóricos
    f2 = sp.diff(f_expr, x, 2)
    f4 = sp.diff(f_expr, x, 4)

    f2_func = sp.lambdify(x, f2, modules=["math"])
    f4_func = sp.lambdify(x, f4, modules=["math"])

    # Métodos numéricos
    Int1 = Int_Trap(f, b, a, n)
    Int2 = Int_Simpson(f, b, a, n)
    Int3 = gauss_legendre(f, b, a, n)

    # Constantes de error
    M2 = max(abs(f2_func(x)) for x in np.linspace(a, b, 200))
    M4 = max(abs(f4_func(x)) for x in np.linspace(a, b, 200))

    h = (b - a) / n

    # Errores teóricos
    E_trap_teo = -(b-a) * h**2 * M2 / 12
    E_simp_teo = -(b-a) * h**4 * M4 / 180
    orden_gauss = "O(h^2n)"   # Orden general de Gauss


    # Mostrar resultados

    print(f"\nFunción {i+1} Integrada por trapecio : {Int1}, con un Error de: {E_trap_teo}")
    print(f"Función {i+1} Integrada por Simpson : {Int2}, con un Error de: {E_simp_teo}")
    print(f"Función {i+1} Integrada por Gauss-Legendre : {Int3}, con un Error de: {orden_gauss}")



Función 1 Integrada por trapecio : 0.6801636890911208, con un Error de: -0.0041554913488750225
Función 1 Integrada por Simpson : 0.6827109757132984, con un Error de: -5.194233016385103e-05
Función 1 Integrada por Gauss-Legendre : 0.6826894921356617, con un Error de: O(h^2n)

Función 2 Integrada por trapecio : 3.1357500211022753, con un Error de: -0.055646815282314306
Función 2 Integrada por Simpson : 3.1225220363811754, con un Error de: -0.002318043710600276
Función 2 Integrada por Gauss-Legendre : 3.1226601971965793, con un Error de: O(h^2n)

Función 3 Integrada por trapecio : 1.0, con un Error de: -0.0009464348715932128
Función 3 Integrada por Simpson : 1.0, con un Error de: -5.54174420836343e-06
Función 3 Integrada por Gauss-Legendre : 0.9999999999999998, con un Error de: O(h^2n)
